# LeetCode 828 — Count Unique Characters of All Substrings of a Given String bằng Python

- Link: https://leetcode.com/problems/count-unique-characters-of-all-substrings-of-a-given-string/
- Độ khó: **Hard**
- Chủ đề: Kỹ thuật đóng góp (contribution technique), Vị trí xuất hiện trước/sau, Quy hoạch động theo điểm kết thúc, Đếm tổ hợp

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def uniqueLetterString(self, s: str) -> int:
```

## 1. Giới thiệu & mục tiêu học

Định nghĩa `countUniqueChars(t)` = số ký tự xuất hiện **đúng một lần** trong chuỗi `t`. Đề yêu cầu tính tổng `countUniqueChars` trên **mọi chuỗi con** (liên tiếp) của `s`.

Chuỗi dài `10⁵` có khoảng `5·10⁹` chuỗi con — không thể duyệt hết. Bài này là ví dụ kinh điển của **kỹ thuật đóng góp**: thay vì hỏi "mỗi chuỗi con đóng góp bao nhiêu?", ta hỏi ngược lại "**mỗi vị trí `i` được tính là ký tự duy nhất trong bao nhiêu chuỗi con?**". Câu trả lời chỉ phụ thuộc vào lần xuất hiện **trước** và **sau** của cùng ký tự.

Sau notebook này bạn sẽ:

1. Viết brute force `O(n³)` và cải tiến `O(n²)` bằng cách mở rộng dần chuỗi con.
2. Hiểu kỹ thuật **đảo chiều tổng** (đổi thứ tự lấy tổng): tổng theo chuỗi con = tổng theo vị trí.
3. Đếm số chuỗi con chứa `i` mà không chứa ký tự cùng loại khác: `(i − prev) · (next − i)`.
4. Cài lời giải `O(n)` bằng DP theo điểm kết thúc với hai mảng "lần xuất hiện gần nhất".
5. Nhận dạng họ bài "tổng trên mọi chuỗi con / mảng con".

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho chuỗi `s` chỉ gồm chữ in hoa `A–Z`. Với mỗi chuỗi con `t = s[l..r]` (mọi cặp `0 <= l <= r < n`, kể cả các chuỗi con giống nhau ở vị trí khác nhau), tính `countUniqueChars(t)` = số ký tự xuất hiện đúng một lần trong `t`. Trả về tổng của tất cả các giá trị đó. Đề đảm bảo kết quả vừa số nguyên 32 bit.

**Ví dụ 1**

```text
Input: s = "ABC"
Output: 10
Giải thích: các chuỗi con "A","B","C","AB","BC","ABC" — mọi ký tự đều duy nhất:
1 + 1 + 1 + 2 + 2 + 3 = 10.
```

**Ví dụ 2**

```text
Input: s = "ABA"
Output: 8
Giải thích: giống ví dụ 1 nhưng countUniqueChars("ABA") = 1 (chỉ có B).
```

**Ví dụ 3**

```text
Input: s = "LEETCODE"
Output: 92
```

**Ràng buộc**

- `1 <= s.length <= 10⁵`
- `s` chỉ gồm chữ cái in hoa tiếng Anh.

In [1]:
from typing import List

EXAMPLES = [
    ("ABC", 10),
    ("ABA", 8),
    ("LEETCODE", 92),
    ("A", 1),
    ("AA", 2),        # "A", "A", "AA"(0)
    ("AAA", 3),
]

def count_unique_chars(t):
    from collections import Counter
    return sum(1 for v in Counter(t).values() if v == 1)

print(count_unique_chars("ABA"), count_unique_chars("LEETCODE"))

1 5


## 3. Kiến thức nền

### 3.1. Số chuỗi con

Chuỗi độ dài `n` có `n(n+1)/2` chuỗi con (chọn `l <= r`). Với `n = 10⁵` là khoảng `5·10⁹`.

In [2]:
s = "ABCD"
subs = [s[l:r + 1] for l in range(len(s)) for r in range(l, len(s))]
print(len(subs), len(s) * (len(s) + 1) // 2, subs)

10 10 ['A', 'AB', 'ABC', 'ABCD', 'B', 'BC', 'BCD', 'C', 'CD', 'D']


### 3.2. `collections.Counter` và mảng đếm 26 ô

`Counter(t)` đếm số lần mỗi ký tự. Với bảng chữ cái cố định, mảng 26 ô (`ord(ch) - 65`) nhanh hơn.

In [3]:
from collections import Counter

print(Counter("LEETCODE"))
cnt = [0] * 26
for ch in "LEETCODE":
    cnt[ord(ch) - 65] += 1
print({chr(65 + i): v for i, v in enumerate(cnt) if v})

Counter({'E': 3, 'L': 1, 'T': 1, 'C': 1, 'O': 1, 'D': 1})
{'C': 1, 'D': 1, 'E': 3, 'L': 1, 'O': 1, 'T': 1}


### 3.3. Số chuỗi con chứa một vị trí trong một đoạn

Trong đoạn `(L, R)` (không kể `L`, `R`), số chuỗi con chứa vị trí `i` là `(i − L) · (R − i)`: chọn đầu trái trong `L+1..i` (`i − L` cách) và đầu phải trong `i..R−1` (`R − i` cách).

In [4]:
L, R, i = -1, 5, 2          # đoạn 0..4, vị trí 2
cnt_direct = sum(1 for l in range(L + 1, R) for r in range(l, R) if l <= i <= r)
print(cnt_direct, (i - L) * (R - i))

9 9


### 3.4. Đổi thứ tự lấy tổng

`Σ_{chuỗi con t} (số vị trí duy nhất trong t) = Σ_{vị trí i} (số chuỗi con mà s[i] là duy nhất)`. Hai cách đếm cùng một tập cặp `(t, i)`.

In [5]:
s = "ABA"
pairs_by_sub = sum(count_unique_chars(s[l:r + 1]) for l in range(3) for r in range(l, 3))
pairs_by_pos = 0
for i in range(3):
    for l in range(i + 1):
        for r in range(i, 3):
            if s[l:r + 1].count(s[i]) == 1:
                pairs_by_pos += 1
print(pairs_by_sub, pairs_by_pos)

8 8


### 3.5. Lưu vị trí xuất hiện theo ký tự

`defaultdict(list)` gom chỉ số theo từng ký tự; thêm lính canh `-1` đầu và `n` cuối để công thức gọn.

In [6]:
from collections import defaultdict

pos = defaultdict(list)
for i, ch in enumerate("LEETCODE"):
    pos[ch].append(i)
print(dict(pos))

{'L': [0], 'E': [1, 2, 7], 'T': [3], 'C': [4], 'O': [5], 'D': [6]}


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Tính tay `"ABA"` theo chuỗi con

| chuỗi con | unique |
|---|---|
| A | 1 |
| B | 1 |
| A | 1 |
| AB | 2 |
| BA | 2 |
| ABA | 1 (chỉ B) |
| **tổng** | **8** |

### 4.2. Tính tay `"ABA"` theo vị trí

- `i = 0` (`A`): A tiếp theo ở 2. Chuỗi con chứa 0 và không chứa 2: `l = 0`, `r ∈ {0, 1}` → 2.
- `i = 1` (`B`): không có B khác. `l ∈ {0,1}`, `r ∈ {1,2}` → 4.
- `i = 2` (`A`): A trước ở 0. `l ∈ {1, 2}`, `r = 2` → 2.

Tổng `2 + 4 + 2 = 8` ✔.

### 4.3. Công thức

Với vị trí `i`, gọi `prev` = vị trí gần nhất **trước** `i` có cùng ký tự (hoặc `−1`), `next` = vị trí gần nhất **sau** (hoặc `n`). Thì `s[i]` duy nhất trong `s[l..r]` ⇔ `prev < l <= i <= r < next`. Số cặp `(l, r)`: `(i − prev) · (next − i)`.

In [7]:
def contribution_table(s):
    n = len(s)
    rows = []
    for i, ch in enumerate(s):
        prev = max([j for j in range(i) if s[j] == ch], default=-1)
        nxt = min([j for j in range(i + 1, n) if s[j] == ch], default=n)
        rows.append((i, ch, prev, nxt, (i - prev) * (nxt - i)))
    return rows

for row in contribution_table("ABA"):
    print("i=%d ch=%s prev=%d next=%d đóng góp=%d" % row)

i=0 ch=A prev=-1 next=2 đóng góp=2
i=1 ch=B prev=-1 next=3 đóng góp=4
i=2 ch=A prev=0 next=3 đóng góp=2


### 4.4. Thêm ví dụ tính tay: `"ABCB"`

Theo vị trí (lính canh `prev = −1`, `next = n = 4`):

| i | ch | prev | next | (i − prev) | (next − i) | đóng góp |
|---|---|---|---|---|---|---|
| 0 | A | −1 | 4 | 1 | 4 | 4 |
| 1 | B | −1 | 3 | 2 | 2 | 4 |
| 2 | C | −1 | 4 | 3 | 2 | 6 |
| 3 | B | 1 | 4 | 2 | 1 | 2 |
| | | | | | | **16** |

Ý nghĩa của ô `i = 1`: chữ `B` ở vị trí 1 là duy nhất trong các chuỗi con `s[l..r]` với `l ∈ {0, 1}` và `r ∈ {1, 2}` (không được kéo tới vị trí 3 vì ở đó có `B` khác) → `"AB"`, `"ABC"`, `"B"`, `"BC"` — đúng 4 chuỗi con.

Theo DP điểm kết thúc (`f(r)` = tổng unique của các chuỗi con kết thúc tại `r`):

| r | ch | last1 | last2 | + (r − last1) | − (last1 − last2) | f(r) | tổng |
|---|---|---|---|---|---|---|---|
| 0 | A | −1 | −1 | 1 | 0 | 1 | 1 |
| 1 | B | −1 | −1 | 2 | 0 | 3 | 4 |
| 2 | C | −1 | −1 | 3 | 0 | 6 | 10 |
| 3 | B | 1 | −1 | 2 | 2 | 6 | 16 |

Hai cách cho cùng 16 ✔.

In [8]:
for row in contribution_table("ABCB"):
    print("i=%d ch=%s prev=%d next=%d đóng góp=%d" % row)
print("liệt kê trực tiếp:", sum(count_unique_chars("ABCB"[l:r + 1]) for l in range(4) for r in range(l, 4)))

i=0 ch=A prev=-1 next=4 đóng góp=4
i=1 ch=B prev=-1 next=3 đóng góp=4
i=2 ch=C prev=-1 next=4 đóng góp=6
i=3 ch=B prev=1 next=4 đóng góp=2
liệt kê trực tiếp: 16


## 5. Cách 1 — Brute Force: liệt kê mọi chuỗi con, đếm trực tiếp

**Ý tưởng.** Hai vòng chọn `l, r`, với mỗi chuỗi con đếm tần suất và cộng số ký tự có tần suất 1.

In [9]:
def unique_brute(s):
    n = len(s)
    total = 0
    for l in range(n):
        for r in range(l, n):
            total += count_unique_chars(s[l:r + 1])
    return total

for s, exp in EXAMPLES:
    assert unique_brute(s) == exp
print("Brute force: OK")

Brute force: OK


**Độ phức tạp.** `O(n²)` chuỗi con × `O(n)` để đếm → `O(n³)`. Với `n = 10⁵` là `10¹⁵`.

**Nút thắt.** Hai chuỗi con `s[l..r]` và `s[l..r+1]` chỉ khác **một ký tự** — đếm lại từ đầu là lãng phí. Đó là cải tiến đầu tiên (cách 2). Nhưng ngay cả `O(n²)` vẫn quá chậm; ta cần đổi góc nhìn (mục 6).

## 6. Bước đột phá

**Insight 1 — Đổi thứ tự tổng.** Thay vì cộng theo chuỗi con, cộng theo **vị trí**: mỗi vị trí `i` đóng góp số chuỗi con mà trong đó `s[i]` là ký tự duy nhất (không có bản sao nào khác của `s[i]`).

**Insight 2 — Chỉ cần hàng xóm cùng loại.** Điều kiện "không có bản sao khác" chỉ phụ thuộc vào bản sao gần nhất bên trái `prev[i]` và bên phải `next[i]`. Đóng góp `= (i − prev[i]) · (next[i] − i)`.

**Insight 3 — DP theo điểm kết thúc.** Gọi `f(r)` = tổng `countUniqueChars(s[l..r])` với mọi `l <= r`. Khi chuyển từ `r − 1` sang `r` với ký tự `c = s[r]`, gọi `last1` = vị trí gần nhất trước `r` của `c`, `last2` = vị trí gần nhì:

- Các chuỗi con có `l > last1`: `c` mới trở thành duy nhất → `+ (r − last1)`.
- Các chuỗi con có `last2 < l <= last1`: trước đây `c` (tại `last1`) là duy nhất, nay xuất hiện 2 lần → `− (last1 − last2)`.
- Các chuỗi con có `l <= last2`: `c` đã xuất hiện ≥ 2 lần, không đổi.
- Mọi ký tự khác: không đổi.

`f(r) = f(r−1) + (r − last1) − (last1 − last2)`, đáp án `= Σ f(r)`.

**Quy tắc nếu…thì:** *nếu* ký tự chưa xuất hiện *thì* `last1 = last2 = −1`; *nếu* mới xuất hiện một lần *thì* `last2 = −1`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mở rộng chuỗi con từ mỗi điểm đầu, `O(n²)`

Cố định `l`, tăng `r`; duy trì mảng đếm và biến `uniq` = số ký tự có tần suất 1:

- tần suất `0 → 1`: `uniq += 1`
- tần suất `1 → 2`: `uniq -= 1`
- tần suất `≥ 2 → +1`: không đổi

In [10]:
def unique_n2(s):
    n = len(s)
    total = 0
    for l in range(n):
        cnt = [0] * 26
        uniq = 0
        for r in range(l, n):
            k = ord(s[r]) - 65
            cnt[k] += 1
            if cnt[k] == 1:
                uniq += 1
            elif cnt[k] == 2:
                uniq -= 1
            total += uniq
    return total

for s, exp in EXAMPLES:
    assert unique_n2(s) == exp
print("O(n^2) mở rộng: OK")

O(n^2) mở rộng: OK


Code "nộp" cho cách 2 (sẽ **TLE** với `n = 10⁵`, chỉ để tham khảo):

In [11]:
class SolutionN2:
    def uniqueLetterString(self, s: str) -> int:
        n, total = len(s), 0
        for l in range(n):
            cnt = [0] * 26
            uniq = 0
            for r in range(l, n):
                k = ord(s[r]) - 65
                cnt[k] += 1
                if cnt[k] == 1:
                    uniq += 1
                elif cnt[k] == 2:
                    uniq -= 1
                total += uniq
        return total

for s, exp in EXAMPLES:
    assert SolutionN2().uniqueLetterString(s) == exp
print("SolutionN2: OK")

SolutionN2: OK


**Độ phức tạp cách 2:** `O(n²)` thời gian, `O(1)` bộ nhớ phụ (26 ô).

### 7.2. Cách 3 — Kỹ thuật đóng góp với `prev`/`next`

Tính `prev[i]` bằng một lượt trái→phải, `next[i]` bằng một lượt phải→trái, mỗi lượt giữ `last[c]` = vị trí gần nhất đã thấy của ký tự `c`.

In [12]:
def unique_contrib(s):
    n = len(s)
    prev, nxt = [-1] * n, [n] * n
    last = [-1] * 26
    for i, ch in enumerate(s):
        k = ord(ch) - 65
        prev[i] = last[k]
        last[k] = i
    last = [n] * 26
    for i in range(n - 1, -1, -1):
        k = ord(s[i]) - 65
        nxt[i] = last[k]
        last[k] = i
    return sum((i - prev[i]) * (nxt[i] - i) for i in range(n))

for s, exp in EXAMPLES:
    assert unique_contrib(s) == exp
print("Đóng góp prev/next: OK")

Đóng góp prev/next: OK


Một biến thể gọn: gom vị trí theo ký tự với lính canh `[-1, ..., n]`, rồi với mỗi ba vị trí liên tiếp `(a, b, c)` cộng `(b − a)(c − b)`.

Code nộp LeetCode cho cách 3:

In [13]:
class SolutionContrib:
    def uniqueLetterString(self, s: str) -> int:
        n = len(s)
        pos = defaultdict(list)
        for i, ch in enumerate(s):
            pos[ch].append(i)
        total = 0
        for idx in pos.values():
            idx = [-1] + idx + [n]
            for j in range(1, len(idx) - 1):
                total += (idx[j] - idx[j - 1]) * (idx[j + 1] - idx[j])
        return total

for s, exp in EXAMPLES:
    assert SolutionContrib().uniqueLetterString(s) == exp
print("SolutionContrib: OK")

SolutionContrib: OK


**Độ phức tạp cách 3:** `O(n)` thời gian, `O(n)` bộ nhớ (danh sách vị trí).

### 7.3. Cách 4 — DP theo điểm kết thúc, một lượt, `O(1)` bộ nhớ phụ

Giữ hai mảng 26 ô: `last1[c]` (lần gần nhất) và `last2[c]` (lần gần nhì). Mỗi bước cập nhật `cur` (= `f(r)`) theo công thức ở mục 6.

In [14]:
def unique_dp(s):
    last1 = [-1] * 26
    last2 = [-1] * 26
    cur = 0
    total = 0
    for r, ch in enumerate(s):
        k = ord(ch) - 65
        cur += (r - last1[k]) - (last1[k] - last2[k])
        total += cur
        last2[k], last1[k] = last1[k], r
    return total

for s, exp in EXAMPLES:
    assert unique_dp(s) == exp
print("DP điểm kết thúc: OK")

DP điểm kết thúc: OK


Giải thích từng dòng:

- `cur += (r - last1[k])`: các chuỗi con kết thúc tại `r` có đầu trái `l` trong `(last1, r]` — ký tự `s[r]` là duy nhất.
- `- (last1[k] - last2[k])`: các chuỗi con có `l` trong `(last2, last1]` — trước đó `s[last1]` là duy nhất, nay bị "phá" bởi `s[r]`.
- `total += cur`: cộng `f(r)`.
- Cập nhật `last2 ← last1`, `last1 ← r`.

Khi ký tự mới lần đầu: `last1 = last2 = −1` → `+(r+1) − 0`. Đúng: mọi `l ∈ [0, r]` đều có `s[r]` duy nhất.

Code nộp LeetCode cho cách 4:

In [15]:
class SolutionDP:
    def uniqueLetterString(self, s: str) -> int:
        last1 = [-1] * 26
        last2 = [-1] * 26
        cur = total = 0
        for r, ch in enumerate(s):
            k = ord(ch) - 65
            cur += (r - last1[k]) - (last1[k] - last2[k])
            total += cur
            last2[k], last1[k] = last1[k], r
        return total

for s, exp in EXAMPLES:
    assert SolutionDP().uniqueLetterString(s) == exp
print("SolutionDP: OK")

SolutionDP: OK


**Độ phức tạp cách 4:** `O(n)` thời gian, `O(1)` bộ nhớ phụ (52 số nguyên).

### 7.4. Cách 5 — Đóng góp tính "trễ" một lượt

Kết hợp ý tưởng cách 3 với một lượt duy nhất: khi gặp lần xuất hiện mới `r` của ký tự `c`, ta đã biết `next` của lần xuất hiện **trước** (`last1[c]`) chính là `r` → cộng đóng góp của `last1[c]` ngay lúc đó. Cuối cùng cộng đóng góp các lần xuất hiện cuối với `next = n`.

In [16]:
def unique_lazy(s):
    n = len(s)
    last1 = [-1] * 26          # lần gần nhất
    last2 = [-1] * 26          # lần trước đó
    total = 0
    for r, ch in enumerate(s):
        k = ord(ch) - 65
        if last1[k] != -1:
            total += (last1[k] - last2[k]) * (r - last1[k])   # đóng góp của vị trí last1
        last2[k], last1[k] = last1[k], r
    for k in range(26):
        if last1[k] != -1:
            total += (last1[k] - last2[k]) * (n - last1[k])
    return total

for s, exp in EXAMPLES:
    assert unique_lazy(s) == exp
print("Đóng góp một lượt: OK")

Đóng góp một lượt: OK


Code nộp LeetCode cho cách 5:

In [17]:
class SolutionLazy:
    def uniqueLetterString(self, s: str) -> int:
        n = len(s)
        last1, last2 = [-1] * 26, [-1] * 26
        total = 0
        for r, ch in enumerate(s):
            k = ord(ch) - 65
            if last1[k] != -1:
                total += (last1[k] - last2[k]) * (r - last1[k])
            last2[k], last1[k] = last1[k], r
        for k in range(26):
            if last1[k] != -1:
                total += (last1[k] - last2[k]) * (n - last1[k])
        return total

for s, exp in EXAMPLES:
    assert SolutionLazy().uniqueLetterString(s) == exp
print("SolutionLazy: OK")

SolutionLazy: OK


**Độ phức tạp cách 5:** `O(n + 26)` thời gian, `O(1)` bộ nhớ phụ.

## 8. TRACE từng bước

### 8.1. Trace cách 4 (DP) trên `"LEETCODE"`

In [18]:
def trace_dp(s):
    last1 = [-1] * 26
    last2 = [-1] * 26
    cur = total = 0
    print(f"{'r':>2} {'ch':>2} {'last1':>5} {'last2':>5} {'+mới':>5} {'-phá':>5} {'f(r)':>5} {'tổng':>5}")
    for r, ch in enumerate(s):
        k = ord(ch) - 65
        add, sub = r - last1[k], last1[k] - last2[k]
        cur += add - sub
        total += cur
        print(f"{r:>2} {ch:>2} {last1[k]:>5} {last2[k]:>5} {add:>5} {sub:>5} {cur:>5} {total:>5}")
        last2[k], last1[k] = last1[k], r
    return total

trace_dp("LEETCODE")

 r ch last1 last2  +mới  -phá  f(r)  tổng
 0  L    -1    -1     1     0     1     1
 1  E    -1    -1     2     0     3     4
 2  E     1    -1     1     2     2     6
 3  T    -1    -1     4     0     6    12
 4  C    -1    -1     5     0    11    23
 5  O    -1    -1     6     0    17    40
 6  D    -1    -1     7     0    24    64
 7  E     2     1     5     1    28    92


92

Đọc bảng: tại `r = 2` (chữ `E` thứ hai), `last1 = 1`, `last2 = −1`: thêm `2 − 1 = 1` chuỗi con mới (`"E"` tại 2), nhưng phá `1 − (−1) = 2` chuỗi con (`"LEE"`, `"EE"` — trong đó chữ E ở vị trí 1 từng là duy nhất trong `"LE"`, `"E"`, nay mở rộng thành `"LEE"`, `"EE"` thì không còn). `f(2) = f(1) − 1`.

### 8.2. Trace cách 3 (đóng góp) trên `"LEETCODE"`

In [19]:
def trace_contrib(s):
    n = len(s)
    total = 0
    print(f"{'i':>2} {'ch':>2} {'prev':>4} {'next':>4} {'trái':>4} {'phải':>4} {'góp':>4}")
    for i, ch, prev, nxt, c in contribution_table(s):
        total += c
        print(f"{i:>2} {ch:>2} {prev:>4} {nxt:>4} {i - prev:>4} {nxt - i:>4} {c:>4}")
    print("tổng =", total)
    return total

trace_contrib("LEETCODE")

 i ch prev next trái phải  góp
 0  L   -1    8    1    8    8
 1  E   -1    2    2    1    2
 2  E    1    7    1    5    5
 3  T   -1    8    4    5   20
 4  C   -1    8    5    4   20
 5  O   -1    8    6    3   18
 6  D   -1    8    7    2   14
 7  E    2    8    5    1    5
tổng = 92


92

Bảng trạng thái (cách 3):

| i | ch | prev | next | (i−prev)·(next−i) |
|---|---|---|---|---|
| 0 | L | −1 | 8 | 1·8 = 8 |
| 1 | E | −1 | 2 | 2·1 = 2 |
| 2 | E | 1 | 7 | 1·5 = 5 |
| 3 | T | −1 | 8 | 4·5 = 20 |
| 4 | C | −1 | 8 | 5·4 = 20 |
| 5 | O | −1 | 8 | 6·3 = 18 |
| 6 | D | −1 | 8 | 7·2 = 14 |
| 7 | E | 2 | 8 | 5·1 = 5 |
| | | | | **92** |

## 9. Code nộp LeetCode

Chọn cách 4 (DP một lượt): ngắn, `O(n)`, `O(1)` bộ nhớ phụ.

In [20]:
from typing import List

class Solution:
    def uniqueLetterString(self, s: str) -> int:
        last1 = [-1] * 26          # vị trí gần nhất của mỗi chữ
        last2 = [-1] * 26          # vị trí gần nhì
        cur = 0                    # f(r) = tổng countUniqueChars của các chuỗi con kết thúc tại r
        total = 0
        for r, ch in enumerate(s):
            k = ord(ch) - 65
            cur += (r - last1[k]) - (last1[k] - last2[k])
            total += cur
            last2[k], last1[k] = last1[k], r
        return total

for s, exp in EXAMPLES:
    assert Solution().uniqueLetterString(s) == exp
print("Solution: OK")

Solution: OK


**Giải thích từng dòng của `Solution`:**

| dòng | ý nghĩa |
|---|---|
| `last1 = [-1] * 26` | vị trí gần nhất của mỗi chữ (chưa thấy = −1) |
| `last2 = [-1] * 26` | vị trí gần nhì |
| `cur` | `f(r)`: tổng unique của mọi chuỗi con **kết thúc tại** `r` |
| `cur += (r - last1[k])` | chuỗi con có `l > last1`: chữ mới trở thành duy nhất |
| `- (last1[k] - last2[k])` | chuỗi con có `last2 < l <= last1`: chữ ở `last1` mất tư cách duy nhất |
| `total += cur` | cộng mọi `f(r)` |
| `last2[k], last1[k] = last1[k], r` | dời "cửa sổ" hai lần xuất hiện gần nhất |

Kiểm tra nhanh kết quả lớn nhất có thể (để yên tâm về kiểu số trong ngôn ngữ khác): chuỗi 10⁵ ký tự với 26 chữ cái xoay vòng.

In [21]:
worst = ''.join(chr(65 + i % 26) for i in range(10 ** 5))
val = Solution().uniqueLetterString(worst)
print(val, "< 2^31?", val < 2 ** 31, "| < 2^63?", val < 2 ** 63)

67583100 < 2^31? True | < 2^63? True


## 10. Vì sao thuật toán đúng

**Cách 3 (đóng góp).** Gọi `X` = tập cặp `(t, i)` sao cho chuỗi con `t = s[l..r]` chứa vị trí `i` và `s[i]` xuất hiện đúng một lần trong `t`. Đáp án `= |X|` (đếm theo `t`). Đếm theo `i`: `s[i]` duy nhất trong `s[l..r]` ⇔ `l <= i <= r` và không có `j ≠ i` trong `[l, r]` với `s[j] = s[i]` ⇔ `prev[i] < l <= i` và `i <= r < next[i]` (vì `prev`/`next` là bản sao **gần nhất**, mọi bản sao khác còn xa hơn). Số cặp `(l, r)` thoả: `(i − prev[i]) · (next[i] − i)`.

**Cách 4 (DP).** Chứng minh công thức chuyển `f(r) = f(r−1) + (r − last1) − (last1 − last2)`:

Mỗi chuỗi con kết thúc tại `r` là `s[l..r]`, với `l <= r − 1` nó là `s[l..r−1]` thêm ký tự `c = s[r]`. Thêm `c` chỉ ảnh hưởng tới **số lần xuất hiện của `c`**:

- `l > last1`: trong `s[l..r−1]` không có `c` → sau khi thêm, `c` xuất hiện 1 lần → unique `+1`. Có `r − 1 − last1` giá trị `l` như vậy, cộng thêm chuỗi con mới `s[r..r]` (unique = 1) → tổng `+ (r − last1)`.
- `last2 < l <= last1`: trong `s[l..r−1]` có đúng một `c` (tại `last1`) → sau khi thêm thành 2 lần → unique `−1`. Có `last1 − last2` giá trị.
- `l <= last2`: đã có ≥ 2 lần `c` → không đổi.

Các ký tự khác không đổi số lần xuất hiện. Cộng lại ra công thức. `total = Σ_r f(r)` = tổng trên mọi chuỗi con.

**Cách 5** là cách 3 nhưng biết `next[last1]` đúng lúc gặp `r`.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n³)` | `O(n)` |
| 2. Mở rộng từ mỗi `l` | `O(n²)` | `O(1)` |
| 3. Đóng góp prev/next | `O(n)` | `O(n)` |
| 4. DP điểm kết thúc | `O(n)` | `O(1)` |
| 5. Đóng góp một lượt | `O(n)` | `O(1)` |

Kết quả lớn nhất: mỗi vị trí đóng góp tối đa `≈ (n/2)²` khi ký tự đó xuất hiện một lần ở giữa; với 26 chữ cái tổng cỡ `n²·hằng số` — Python dùng số nguyên lớn nên không lo tràn.

In [22]:
import random, time

rnd = random.Random(828)
big = ''.join(rnd.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ") for _ in range(10 ** 5))
for name, fn in [("contrib", unique_contrib), ("dp", unique_dp), ("lazy", unique_lazy),
                 ("SolutionContrib", lambda x: SolutionContrib().uniqueLetterString(x)),
                 ("Solution", lambda x: Solution().uniqueLetterString(x))]:
    t0 = time.perf_counter()
    res = fn(big)
    print(f"{name:16}: {res}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")

t0 = time.perf_counter(); unique_n2(big[:2000]); print(f"O(n^2) với n=2000: {(time.perf_counter() - t0) * 1000:.0f} ms")

contrib         : 67298103  (27 ms)


dp              : 67298103  (13 ms)


lazy            : 67298103  (14 ms)


SolutionContrib : 67298103  (20 ms)
Solution        : 67298103  (15 ms)


O(n^2) với n=2000: 213 ms


## 12. Bộ test

In [23]:
methods = {
    "brute": unique_brute,
    "n2": unique_n2,
    "contrib": unique_contrib,
    "dp": unique_dp,
    "lazy": unique_lazy,
    "SolutionN2": lambda x: SolutionN2().uniqueLetterString(x),
    "SolutionContrib": lambda x: SolutionContrib().uniqueLetterString(x),
    "SolutionDP": lambda x: SolutionDP().uniqueLetterString(x),
    "SolutionLazy": lambda x: SolutionLazy().uniqueLetterString(x),
    "Solution": lambda x: Solution().uniqueLetterString(x),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    ("AB", 4),
    ("ABAB", 12),
    ("Z" * 10, 10),                       # chỉ các chuỗi con độ dài 1 có unique
    ("ABCDEFGHIJKLMNOPQRSTUVWXYZ", sum((i + 1) * (26 - i) for i in range(26))),
]
for name, fn in methods.items():
    for s, exp in edge:
        assert fn(s) == exp, (name, s, fn(s), exp)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed: đối chiếu brute force
for seed in range(600):
    r_ = random.Random(seed)
    alpha = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"[: r_.randint(1, 6)]
    s = ''.join(r_.choice(alpha) for _ in range(r_.randint(1, 14)))
    ref = unique_brute(s)
    for name, fn in methods.items():
        assert fn(s) == ref, (name, s)
print("600 chuỗi ngẫu nhiên: khớp brute force")

# 12.3 Chuỗi vừa: các cách O(n) khớp O(n^2)
for seed in range(20):
    r_ = random.Random(5000 + seed)
    s = ''.join(r_.choice("ABCDE") for _ in range(400))
    assert unique_n2(s) == unique_contrib(s) == unique_dp(s) == unique_lazy(s)
print("Chuỗi độ dài 400: các cách O(n) khớp O(n^2)")

Ví dụ + edge cases: OK
600 chuỗi ngẫu nhiên: khớp brute force


Chuỗi độ dài 400: các cách O(n) khớp O(n^2)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đếm số ký tự **khác nhau** thay vì số ký tự **xuất hiện đúng một lần**

**Code sai:**

In [24]:
def wrong_distinct(s):
    n = len(s)
    return sum(len(set(s[l:r + 1])) for l in range(n) for r in range(l, n))   # SAI

print("sai:", wrong_distinct("ABA"), " đúng:", Solution().uniqueLetterString("ABA"))

sai: 9  đúng: 8


**Vì sao sai:** `"ABA"` có 2 ký tự khác nhau nhưng chỉ 1 ký tự duy nhất (`B`). Đây là bài khác (LeetCode 2262 Total Appeal of a String).

**Code đúng:** đếm ký tự có tần suất bằng 1.

### Lỗi 2 — Dùng `next` là vị trí **xa nhất** hoặc quên lính canh

**Code sai:**

In [25]:
def wrong_sentinel(s):
    n = len(s)
    pos = defaultdict(list)
    for i, ch in enumerate(s):
        pos[ch].append(i)
    total = 0
    for idx in pos.values():
        idx = [0] + idx + [n - 1]                 # SAI: lính canh phải là -1 và n
        for j in range(1, len(idx) - 1):
            total += (idx[j] - idx[j - 1]) * (idx[j + 1] - idx[j])
    return total

print("sai:", wrong_sentinel("ABC"), " đúng:", Solution().uniqueLetterString("ABC"))

sai: 1  đúng: 10


**Vì sao sai:** với lính canh `0` và `n−1`, vị trí `0` có `i − prev = 0` → đóng góp bị triệt tiêu; vị trí cuối cũng vậy.

**Code đúng:** `prev` mặc định `−1`, `next` mặc định `n` (số cách chọn đầu trái là `i − (−1) = i + 1`).

### Lỗi 3 — DP quên trừ phần bị "phá"

**Code sai:**

In [26]:
def wrong_dp_no_sub(s):
    last1 = [-1] * 26
    cur = total = 0
    for r, ch in enumerate(s):
        k = ord(ch) - 65
        cur += r - last1[k]                       # SAI: thiếu - (last1 - last2)
        total += cur
        last1[k] = r
    return total

print("sai:", wrong_dp_no_sub("ABA"), " đúng:", Solution().uniqueLetterString("ABA"))

sai: 9  đúng: 8


**Vì sao sai:** đây chính là công thức đếm **số ký tự khác nhau** (bài 2262)! Khi `c` xuất hiện lần thứ hai trong chuỗi con, lần thứ nhất mất tư cách "duy nhất" — phải trừ.

**Code đúng:** `cur += (r - last1) - (last1 - last2)`.

### Lỗi 4 — Cập nhật `last1` trước `last2`

`last1[k] = r; last2[k] = last1[k]` → `last2` thành `r`, sai. Dùng gán song song `last2[k], last1[k] = last1[k], r`.

In [27]:
def wrong_update_order(s):
    last1, last2 = [-1] * 26, [-1] * 26
    cur = total = 0
    for r, ch in enumerate(s):
        k = ord(ch) - 65
        cur += (r - last1[k]) - (last1[k] - last2[k])
        total += cur
        last1[k] = r
        last2[k] = last1[k]                       # SAI: last2 = r
    return total

print("sai:", wrong_update_order("ABAB"), " đúng:", Solution().uniqueLetterString("ABAB"))

sai: 16  đúng: 12


## 14. Mẹo & nhận dạng pattern

- **"Tổng một đại lượng trên mọi chuỗi con / mảng con"** → hầu như luôn là **kỹ thuật đóng góp**: mỗi phần tử đóng góp bao nhiêu, nhân số chuỗi con mà nó "có hiệu lực".
- Số chuỗi con chứa `i` trong khoảng mở `(L, R)` là `(i − L)(R − i)` — công thức vàng.
- `L`, `R` thường là "phần tử gần nhất cùng loại" (bài này), "phần tử gần nhất lớn/nhỏ hơn" (stack đơn điệu — 907 Sum of Subarray Minimums, 2104 Sum of Subarray Ranges).
- Cách DP theo điểm kết thúc: `f(r)` từ `f(r−1)` bằng cách xét ký tự mới ảnh hưởng tới những chuỗi con nào.
- Bài liên quan: **2262 Total Appeal of a String**, **907 Sum of Subarray Minimums**, **1918**, **2063 Vowels of All Substrings**.

## 15. Biến thể

1. **Tổng số ký tự khác nhau trên mọi chuỗi con** (2262): đóng góp `(i − prev) · (n − i)` (chỉ cần là lần xuất hiện **đầu tiên** trong chuỗi con).
2. **Ký tự xuất hiện đúng `k` lần**: xét mỗi nhóm `k` lần xuất hiện liên tiếp `p_j..p_{j+k−1}`, đóng góp `(p_j − p_{j−1}) · (p_{j+k} − p_{j+k−1})`.
3. **Bảng chữ cái tổng quát** (Unicode): thay mảng 26 bằng `dict`.
4. **Mảng số nguyên thay vì chuỗi**: y hệt, dùng `dict` cho `last`.
5. **Có truy vấn cập nhật ký tự**: duy trì `SortedList` vị trí cho từng ký tự, cập nhật đóng góp cục bộ `O(log n)`.

Minh hoạ biến thể 1 và 2:

In [28]:
def total_distinct(s):
    n, last, total = len(s), {}, 0
    for i, ch in enumerate(s):
        total += (i - last.get(ch, -1)) * (n - i)
        last[ch] = i
    return total

def exactly_k_total(s, k):
    n = len(s)
    pos = defaultdict(list)
    for i, ch in enumerate(s):
        pos[ch].append(i)
    total = 0
    for idx in pos.values():
        p = [-1] + idx + [n]
        for j in range(1, len(p) - k):
            total += (p[j] - p[j - 1]) * (p[j + k] - p[j + k - 1])
    return total

def exactly_k_brute(s, k):
    n = len(s)
    return sum(sum(1 for v in Counter(s[l:r + 1]).values() if v == k) for l in range(n) for r in range(l, n))

print(total_distinct("ABA"), wrong_distinct("ABA"))
for k in (1, 2, 3):
    assert exactly_k_total("ABAACBAB", k) == exactly_k_brute("ABAACBAB", k)
print("đúng k lần:", [exactly_k_total("ABAACBAB", k) for k in (1, 2, 3)])

9 9
đúng k lần: [51, 17, 9]


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | mọi chuỗi con, đếm `Counter` | `O(n³)` | `O(n)` | đối chiếu |
| 2. Mở rộng từ `l` | duy trì `uniq` khi tăng `r` | `O(n²)` | `O(1)` | `n ≤ 5000` |
| 3. Đóng góp prev/next | `(i−prev)(next−i)` | `O(n)` | `O(n)` | dễ giải thích, dễ mở rộng |
| 4. DP điểm kết thúc | `f(r)` từ `f(r−1)` | `O(n)` | `O(1)` | **nộp bài** |
| 5. Đóng góp một lượt | tính đóng góp khi biết `next` | `O(n)` | `O(1)` | khi muốn một lượt + công thức đóng góp |

## 17. Bài tập

**Bài 1 (đoán output).** `uniqueLetterString("AB")`? `"ABAB"`? `"AAAA"`?

**Bài 2 (tính tay).** Lập bảng `prev/next/đóng góp` cho `"ABCAB"` và tính tổng.

**Bài 3 (điền TODO).** Hoàn thiện `contribution_list(s)` trả về danh sách đóng góp của từng vị trí, chỉ dùng **một** lượt duyệt (gợi ý cách 5).

**Bài 4 (tự viết lại).** Viết lời giải cho **mảng số nguyên** `nums` (giá trị tới `10⁹`): tổng số phần tử xuất hiện đúng một lần trên mọi mảng con.

**Bài 5 (mini challenge).** Tính tổng `countUniqueChars` chỉ trên các chuỗi con có **độ dài chẵn**.

In [29]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def contribution_list_todo(s):
    # TODO: contrib = [0]*n; khi gặp r có last1, contrib[last1] = (last1-last2)*(r-last1)
    pass

# Bài 4 — TODO
def unique_in_array_todo(nums):
    # TODO: như Solution nhưng last1/last2 là dict
    pass

# Bài 5 — TODO
def unique_even_length_todo(s):
    # TODO: đếm cặp (l, r) với prev < l <= i <= r < next và (r - l + 1) chẵn
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: liệt kê chuỗi con.
- Mức 2: `"AB"`: 1 + 1 + 2.
- Mức 3: `4`, `12`, `4` (chỉ 4 chuỗi con độ dài 1).

**Bài 2.**
- Mức 1: vị trí A: `[0, 3]`, B: `[1, 4]`, C: `[2]`.
- Mức 2: đóng góp: `i=0: 1·3`, `i=1: 2·3`, `i=2: 3·3`, `i=3: 3·2`, `i=4: 3·1`.
- Mức 3: `3 + 6 + 9 + 6 + 3 = 27`.

**Bài 3.**
- Mức 1: giữ `last1`, `last2`.
- Mức 2: khi gặp lần mới `r` của `c`, `next` của `last1[c]` là `r`.
- Mức 3: cuối cùng xử lý các `last1[c]` còn lại với `next = n`.

**Bài 4.**
- Mức 1: thay `ord(ch) - 65` bằng khoá `dict`.
- Mức 2: `last1.get(x, -1)`.
- Mức 3: kiểm tra bằng brute force với `Counter`.

**Bài 5.**
- Mức 1: với vị trí `i`, đếm cặp `(l, r)` trong hình chữ nhật `[prev+1, i] × [i, next−1]` có `r − l` lẻ.
- Mức 2: số `l` chẵn/lẻ trong đoạn và số `r` chẵn/lẻ trong đoạn; `r − l` lẻ ⇔ khác tính chẵn lẻ.
- Mức 3: `lEven·rOdd + lOdd·rEven`.

## 19. Đáp án tham khảo

In [30]:
# Bài 1
assert [Solution().uniqueLetterString(x) for x in ("AB", "ABAB", "AAAA")] == [4, 12, 4]

# Bài 2
assert Solution().uniqueLetterString("ABCAB") == unique_brute("ABCAB") == 27

# Bài 3
def contribution_list(s):
    n = len(s)
    contrib = [0] * n
    last1, last2 = {}, {}
    for r, ch in enumerate(s):
        if ch in last1:
            p = last1[ch]
            contrib[p] = (p - last2.get(ch, -1)) * (r - p)
        last2[ch], last1[ch] = last1.get(ch, -1), r
    for ch, p in last1.items():
        contrib[p] = (p - last2.get(ch, -1)) * (n - p)
    return contrib

assert contribution_list("LEETCODE") == [8, 2, 5, 20, 20, 18, 14, 5]

# Bài 4
def unique_in_array(nums):
    last1, last2 = {}, {}
    cur = total = 0
    for r, x in enumerate(nums):
        a, b = last1.get(x, -1), last2.get(x, -1)
        cur += (r - a) - (a - b)
        total += cur
        last2[x], last1[x] = a, r
    return total

def unique_in_array_brute(nums):
    n = len(nums)
    return sum(sum(1 for v in Counter(nums[l:r + 1]).values() if v == 1) for l in range(n) for r in range(l, n))

for seed in range(200):
    r_ = random.Random(seed)
    arr = [r_.choice([10 ** 9, -5, 7, 7, 0]) for _ in range(r_.randint(1, 12))]
    assert unique_in_array(arr) == unique_in_array_brute(arr)

# Bài 5
def unique_even_length(s):
    n = len(s)
    prev, nxt, last = [-1] * n, [n] * n, {}
    for i, ch in enumerate(s):
        prev[i] = last.get(ch, -1); last[ch] = i
    last = {}
    for i in range(n - 1, -1, -1):
        nxt[i] = last.get(s[i], n); last[s[i]] = i

    def count_parity(lo, hi):          # số chẵn, số lẻ trong [lo, hi]
        tot = hi - lo + 1
        evens = hi // 2 - (lo - 1) // 2 if lo > 0 else hi // 2 + 1
        return evens, tot - evens

    total = 0
    for i in range(n):
        le, lo_ = count_parity(prev[i] + 1, i)
        re, ro = count_parity(i, nxt[i] - 1)
        total += le * ro + lo_ * re
    return total

def unique_even_length_brute(s):
    n = len(s)
    return sum(count_unique_chars(s[l:r + 1]) for l in range(n) for r in range(l, n) if (r - l + 1) % 2 == 0)

for seed in range(300):
    r_ = random.Random(seed)
    s = ''.join(r_.choice("ABC") for _ in range(r_.randint(1, 12)))
    assert unique_even_length(s) == unique_even_length_brute(s), s
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (DP điểm kết thúc):**

```python
last1 = [-1]*26; last2 = [-1]*26; cur = total = 0
for r, ch in enumerate(s):
    k = ord(ch) - 65
    cur += (r - last1[k]) - (last1[k] - last2[k])
    total += cur
    last2[k], last1[k] = last1[k], r
return total
```

**Template (đóng góp):** `total += (i - prev[i]) * (next[i] - i)` với `prev = -1`, `next = n` mặc định.

**Checklist trước khi submit:**

- [ ] Đếm ký tự xuất hiện **đúng một lần**, không phải số ký tự khác nhau.
- [ ] Lính canh `prev = −1`, `next = n`.
- [ ] DP có **cả hai** số hạng `+ (r − last1)` và `− (last1 − last2)`.
- [ ] Cập nhật `last2, last1 = last1, r` **song song**.
- [ ] Thử nhanh: `"ABC" → 10`, `"ABA" → 8`, `"LEETCODE" → 92`.